In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, random_split
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader as PyGDataLoader

import numpy as np
import pandas as pd
from rdkit import Chem
from scipy.signal.windows import triang
from sklearn.metrics import mean_squared_error

In [ ]:
class Config:
    TRAIN_RATIO = 0.9  # 검증 세트 비율 (1 - TRAIN_RATIO)
    BATCH_SIZE = 32

    # Inhibition(%)을 p-value로 변환하기 위한 설정
    EPSILON = 1e-9 # log(0) 방지
    
    DESC_DIM = 18 # 우리가 추가할 기술자의 수
    GNN_EMBED_DIM = 256 # 모델 용량을 일단 원래대로 되돌려서 시작
    HIDDEN_DIM = 512
    
    # LDS & FDS 관련 (p-value 범위에 맞게 조정)
    LDS_KERNEL = 'gaussian'
    LDS_KS = 9
    LDS_SIGMA = 5
    FDS_FEATURE_DIM = GNN_EMBED_DIM + DESC_DIM 
    FDS_BUCKET_NUM = 100 
    FDS_BUCKET_START = -10.0 # p-value는 0부터 시작
    FDS_START_UPDATE_EPOCH = 2

    # 학습 관련
    LEARNING_RATE = 5e-5
    EPOCHS = 100
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEED = 42
    EARLY_STOPPING_PATIENCE = 100

cfg = Config()
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)

In [ ]:
def normalized_rmse(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    range_y = np.max(y_true) - np.min(y_true)
    return rmse / range_y if range_y != 0 else rmse

def pearson_correlation(y_true, y_pred):
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return np.clip(corr, 0, 1) if not np.isnan(corr) else 0

def competition_score(y_true, y_pred):
    nrmse = min(normalized_rmse(y_true, y_pred), 1)
    pearson = pearson_correlation(y_true, y_pred)
    return 0.5 * (1 - nrmse) + 0.5 * pearson

In [ ]:
def get_lds_kernel_window(kernel, ks, sigma):
    assert kernel in ['gaussian', 'triang']; half_ks = (ks - 1) // 2
    if kernel == 'gaussian': x = np.arange(-half_ks, half_ks + 1); kernel_window = np.exp(-x**2 / (2 * sigma**2))
    else: kernel_window = triang(ks)
    return torch.from_numpy(kernel_window).float() / sum(kernel_window)

class FDS(nn.Module):
    def __init__(self, feature_dim, bucket_num, bucket_start, start_update, kernel, ks, sigma):
        super(FDS, self).__init__(); self.feature_dim = feature_dim; self.bucket_num = bucket_num; self.bucket_start = bucket_start; self.start_update = start_update
        self.kernel_window = get_lds_kernel_window(kernel, ks, sigma).to(cfg.DEVICE)
        self.register_buffer('running_mean', torch.zeros(bucket_num, feature_dim)); self.register_buffer('running_var', torch.ones(bucket_num, feature_dim))
        self.register_buffer('bucket_counts', torch.zeros(bucket_num)); self.register_buffer('running_mean_last_epoch', torch.zeros(bucket_num, feature_dim))
        self.register_buffer('running_var_last_epoch', torch.ones(bucket_num, feature_dim))
    def reset(self): self.running_mean.zero_(); self.running_var.fill_(1); self.bucket_counts.zero_()
    def update_last_epoch_stats(self, epoch):
        if epoch < self.start_update: return
        self.running_mean_last_epoch.copy_(self.running_mean); self.running_var_last_epoch.copy_(self.running_var); self.reset()
    def _smooth(self, stats, counts):
        stats[counts == 0] = 0; stats = stats.permute(1, 0).unsqueeze(1); kernel = self.kernel_window.view(1, 1, -1)
        padding = (self.kernel_window.size(0) - 1) // 2; smoothed_stats = F.conv1d(stats, weight=kernel, padding=padding, groups=1)
        return smoothed_stats.squeeze(1).permute(1, 0)

    def forward(self, features, labels, epoch):
        if labels is None:
            return features

        if epoch < self.start_update:
            return features

        smoothed_mean = self._smooth(self.running_mean_last_epoch, self.bucket_counts)
        smoothed_var = self._smooth(self.running_var_last_epoch, self.bucket_counts)

        bucket_indices = torch.floor(labels - self.bucket_start).long().clamp(0, self.bucket_num - 1)
        
        std_dev_last = torch.sqrt(self.running_var_last_epoch[bucket_indices]) + 1e-6
        std_dev_smoothed = torch.sqrt(smoothed_var[bucket_indices]) + 1e-6
        calibrated_features = (features - self.running_mean_last_epoch[bucket_indices]) / std_dev_last * std_dev_smoothed + smoothed_mean[bucket_indices]

        if self.training:
            with torch.no_grad():
                detached_features = features.detach()
                for i in range(self.bucket_num):
                    mask = (bucket_indices == i)
                    if mask.sum() > 0:
                        bucket_features = detached_features[mask]
                        n_new = bucket_features.size(0)
                        n_old = self.bucket_counts[i].item()
                        
                        new_total = n_old + n_new
                        self.running_mean[i] = (self.running_mean[i] * n_old + torch.sum(bucket_features, 0)) / new_total
                        self.running_var[i] = (self.running_var[i] * n_old * (n_old > 0) + torch.sum(bucket_features**2, 0)) / new_total - self.running_mean[i]**2
                        self.bucket_counts[i] += n_new
                        
        return calibrated_features

In [ ]:
from rdkit.Chem import Descriptors, Lipinski
def calculate_descriptors(smiles_list):
    """SMILES 리스트로부터 18개의 기술자를 계산하는 함수"""
    descriptor_values = []
    descriptor_names = [
        "MolWt", "MolLogP", "NumHAcceptors", "NumHDonors", "TPSA", 
        "NumRotatableBonds", "NumAromaticRings", "NumHeteroatoms", 
        "FractionCSP3", "NumAliphaticRings", "NumAromaticHeterocycles",
        "NumSaturatedHeterocycles", "NumAliphaticHeterocycles", "HeavyAtomCount",
        "RingCount", "NOCount", "NHOHCount", "NumRadicalElectrons"
    ]
    
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        desc_vals = []
        if mol:
            desc_vals = [
                Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.NumHAcceptors(mol),
                Descriptors.NumHDonors(mol), Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol),
                Descriptors.NumAromaticRings(mol), Descriptors.NumHeteroatoms(mol), Descriptors.FractionCSP3(mol),
                Descriptors.NumAliphaticRings(mol), Lipinski.NumAromaticHeterocycles(mol),
                Lipinski.NumSaturatedHeterocycles(mol), Lipinski.NumAliphaticHeterocycles(mol),
                Descriptors.HeavyAtomCount(mol), Descriptors.RingCount(mol), Descriptors.NOCount(mol),
                Descriptors.NHOHCount(mol), Descriptors.NumRadicalElectrons(mol)
            ]
        else: # 변환 실패 시 0으로 채움
            desc_vals = [0] * len(descriptor_names)
        descriptor_values.append(desc_vals)
        
    return np.array(descriptor_values, dtype=np.float32)

In [ ]:
# RDKit에서 추가적인 모듈 임포트
from rdkit.Chem import rdMolDescriptors

# smiles_to_graph 함수를 아래 코드로 완전히 교체
def smiles_to_graph(smiles):
    """SMILES로부터 원자 특성과 결합 특성을 모두 추출하는 함수"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None

    # --- 1. 원자(Node) 특성 생성 (이전과 동일) ---
    atom_features = []
    for atom in mol.GetAtoms():
        is_cw = int(str(atom.GetChiralTag()) == 'CHI_TETRAHEDRAL_CW')
        is_ccw = int(str(atom.GetChiralTag()) == 'CHI_TETRAHEDRAL_CCW')
        atom_features.append([
            atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(),
            atom.GetHybridization(), float(atom.GetIsAromatic()),
            atom.GetTotalNumHs(), float(atom.IsInRing()), is_cw, is_ccw
        ])
    x = torch.tensor(atom_features, dtype=torch.float32)

    # --- 2. 결합(Edge) 특성 추가 ---
    edge_indices = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_indices.extend([(i, j), (j, i)]) # 양방향 엣지 추가
        
        # 결합 특성 정의 (결합 차수, 컨쥬게이션 여부, 고리 포함 여부)
        bond_type = [
            bond.GetBondTypeAsDouble(),
            float(bond.GetIsConjugated()),
            float(bond.IsInRing())
        ]
        edge_attrs.extend([bond_type, bond_type]) # 양방향 엣지에 동일한 특성 부여

    if len(edge_indices) > 0:
        edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float32)
    else: # 결합이 없는 분자 처리
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 3), dtype=torch.float32) # edge_attr 차원 수(3)에 맞춰야 함

    # Data 객체에 edge_attr 추가
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

class MolDataset(Dataset):
    def __init__(self, df, descriptors, cfg, is_test=False):
        self.df = df
        self.descriptors = torch.tensor(descriptors, dtype=torch.float32) # 기술자 데이터 저장
        self.cfg = cfg
        self.is_test = is_test
        if not self.is_test:
            # --- 수정된 부분 ---
            inhibition = torch.tensor(df['Inhibition'].values, dtype=torch.float32)
            
            # Logit 변환
            # 1. Inhibition을 0~1 사이의 비율(ratio)로 변환
            ratio = inhibition / 100.0
            # 2. log(0) 또는 log(1/(1-1)) 오류를 막기 위해 값을 작은 범위 내로 제한(clipping)
            ratio = torch.clamp(ratio, cfg.EPSILON, 1 - cfg.EPSILON)
            # 3. Logit 변환 적용
            self.logit_values = torch.log(ratio / (1 - ratio))
            self.labels = self.logit_values # self.labels를 새로운 타겟으로 사용
            # --- 여기까지 수정 ---
            
            # LDS는 self.labels를 기반으로 계산되므로 이 부분은 수정할 필요 없음
            self.lds_weights = self._prepare_lds_weights(self.labels, cfg)

    # 수정된 _prepare_lds_weights 함수

    def _prepare_lds_weights(self, labels, cfg):
        min_v, max_v = int(labels.min().floor()), int(labels.max().ceil())
        
        if min_v == max_v:
            max_v += 1
            
        num_bins = max_v - min_v + 1
        label_dist = torch.histc(labels, bins=num_bins, min=min_v, max=max_v)
        lds_kernel = get_lds_kernel_window(cfg.LDS_KERNEL, cfg.LDS_KS, cfg.LDS_SIGMA)
        eff_label_dist = F.conv1d(
            label_dist.unsqueeze(0).unsqueeze(0), 
            lds_kernel.unsqueeze(0).unsqueeze(0), 
            padding=(cfg.LDS_KS - 1) // 2
        ).squeeze()
        weights = 1.0 / (eff_label_dist + 1e-6)
        label_bin_ids = (labels - min_v).long().clamp(0, len(weights) - 1)
        lds_weights = weights[label_bin_ids]
        return lds_weights / torch.sum(lds_weights) * len(labels)
        
    def __len__(self): return len(self.df)
    # 수정된 __getitem__ 메소드

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        smiles = row['Canonical_Smiles']
        graph = smiles_to_graph(smiles)
        if graph is None: return self[(idx + 1) % len(self)]
        
        graph.id = row['ID']
        graph.descriptors = self.descriptors[idx] # Data 객체에 기술자 추가
        
        if not self.is_test:
            graph.y = self.labels[idx]
            graph.inhibition = row['Inhibition']
            graph.lds_weight = self.lds_weights[idx]
        return graph

In [ ]:
from torch_geometric.nn import GINConv as PyG_GINConv, global_add_pool, BatchNorm

class GINConv(torch.nn.Module):
    """
    MolCLR에서 사용하는 GIN Convolution 레이어의 래퍼(wrapper)
    (GINConv + BatchNorm + ReLU)
    """
    def __init__(self, emb_dim):
        super(GINConv, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, 2 * emb_dim), 
            nn.BatchNorm1d(2 * emb_dim), 
            nn.ReLU(), 
            nn.Linear(2 * emb_dim, emb_dim)
        )
        self.conv = PyG_GINConv(self.mlp, train_eps=True)
        self.bn = nn.BatchNorm1d(emb_dim)

    def forward(self, x, edge_index):
        return F.relu(self.bn(self.conv(x, edge_index)))

class MolCLR_Encoder(nn.Module):
    """
    사전 학습된 MolCLR의 GIN 기반 인코더.
    """
    def __init__(self, num_layer=5, emb_dim=300, feat_dim=5, drop_ratio=0.2):
        super(MolCLR_Encoder, self).__init__()
        self.num_layer = num_layer
        self.emb_dim = emb_dim
        self.drop_ratio = drop_ratio
        
        # MolCLR은 원자 번호를 임베딩 레이어에 통과시켜 사용합니다.
        # 우리의 원자 특성은 9차원이지만, MolCLR은 원자 번호(첫 번째 특성)만 사용합니다.
        self.x_embedding = nn.Embedding(119, emb_dim) # 118개 원소 + 1(unknown)

        self.gnns = nn.ModuleList()
        for _ in range(num_layer):
            self.gnns.append(GINConv(emb_dim))

        self.pool = global_add_pool

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # 원자 번호(첫 번째 특성)만 정수형으로 변환하여 임베딩
        x = self.x_embedding(x[:, 0].long())

        for i in range(self.num_layer):
            x = self.gnns[i](x, edge_index)

        graph_embedding = self.pool(x, batch)
        return graph_embedding


class Pignet(nn.Module):
    """인코더가 사전 학습된 MolCLR로 교체된 최종 모델"""
    def __init__(self, cfg):
        super(Pignet, self).__init__()
        
        # 1. MolCLR 인코더 생성 (변경 없음)
        # MolCLR의 임베딩 차원은 300으로 고정되어 있습니다.
        self.gnn_encoder = MolCLR_Encoder(emb_dim=300)
        
        # ==================== 수정된 핵심 부분 ====================
        # --- 사전 학습된 가중치 로드 ---
        # 파일 경로를 공식 리포지토리 경로로 수정
        model_path = 'model.pth' 
        try:
            # 체크포인트 파일 로드
            checkpoint = torch.load(model_path, map_location=cfg.DEVICE)
            
            # 체크포인트는 {'model': ..., 'optimizer': ...} 형태의 딕셔너리
            # 'model' 키에 실제 state_dict가 들어있음
            state_dict = checkpoint['model']
            
            # 'backbone_q.gnn.' 으로 시작하는 키만 추출하여 우리 모델에 맞게 이름 변경
            prefix = 'backbone_q.gnn.'
            encoder_state_dict = {}
            for k, v in state_dict.items():
                if k.startswith(prefix):
                    new_key = k[len(prefix):] # 접두사 제거
                    encoder_state_dict[new_key] = v
            
            # 생성한 state_dict로 가중치 로드
            self.gnn_encoder.load_state_dict(encoder_state_dict)
            print(f"Successfully loaded pre-trained MolCLR weights from {model_path}.")

        except FileNotFoundError:
            print(f"Pre-trained weights not found at {model_path}. Training from scratch.")
        except KeyError:
            print(f"Could not find required keys in the checkpoint at {model_path}. Check the file structure.")
        # ========================================================

        # FDS와 MLP의 입력 차원을 MolCLR 출력에 맞게 수정
        # MolCLR 출력(300) + 기술자(18)
        FDS_INPUT_DIM = 300 + cfg.DESC_DIM
        
        # 2. FDS 모듈 (FDS_INPUT_DIM을 사용하도록 수정)
        self.fds = FDS(
            feature_dim=FDS_INPUT_DIM, 
            bucket_num=cfg.FDS_BUCKET_NUM,
            bucket_start=cfg.FDS_BUCKET_START,
            start_update=cfg.FDS_START_UPDATE_EPOCH,
            kernel=cfg.LDS_KERNEL,
            ks=cfg.LDS_KS,
            sigma=cfg.LDS_SIGMA
        )
        
        # 3. 예측기 헤드 (FDS_INPUT_DIM을 사용하도록 수정)
        self.predictor_head = nn.Sequential(
            nn.Linear(FDS_INPUT_DIM, cfg.HIDDEN_DIM),
            nn.BatchNorm1d(cfg.HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(cfg.HIDDEN_DIM, 1)
        )

    # forward 메소드는 이전과 동일하게 유지
    def forward(self, data, epoch):
        graph_embedding = self.gnn_encoder(data)
        descriptors = data.descriptors.view(data.num_graphs, -1)
        combined_features = torch.cat([graph_embedding, descriptors], dim=1)
        
        labels = data.y if hasattr(data, 'y') else None
        calibrated_features = self.fds(combined_features, labels, epoch)
        return self.predictor_head(calibrated_features).squeeze(1)

    def forward(self, data, epoch):
        # forward 로직은 하이브리드 모델과 동일
        graph_embedding = self.gnn_encoder(data)
        descriptors = data.descriptors.view(data.num_graphs, -1)
        combined_features = torch.cat([graph_embedding, descriptors], dim=1)
        
        labels = data.y if hasattr(data, 'y') else None
        calibrated_features = self.fds(combined_features, labels, epoch)
        return self.predictor_head(calibrated_features).squeeze(1)

In [ ]:
def WeightedMSELoss(predictions, targets, weights): return torch.mean(weights * (predictions - targets)**2)
def WeightedL1Loss(predictions, targets, weights):
    """LDS 가중치를 적용한 L1 Loss (MAE)"""
    loss = torch.abs(predictions - targets) # 제곱 대신 절대값 사용
    weighted_loss = torch.mean(weights * loss)
    return weighted_loss

def train_epoch(model, loader, optimizer, epoch, device):
    model.train(); total_loss = 0
    for batch in loader:
        batch = batch.to(device); optimizer.zero_grad()
        predictions = model(batch, epoch)
        # loss = WeightedMSELoss(predictions, batch.y, batch.lds_weight)
        loss = WeightedL1Loss(predictions, batch.y, batch.lds_weight)
        loss.backward()
        
        # --- 이 부분을 추가하세요 ---
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0) # 그래디언트 클리핑
        
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()

def eval_epoch(model, loader, epoch, device):
    model.eval()
    y_true_list, p_pred_list = [], []
    for batch in loader:
        batch = batch.to(device)
        predictions = model(batch, epoch)
        y_true_list.extend(batch.inhibition.cpu().numpy())
        p_pred_list.extend(predictions.cpu().numpy())
    
    y_true_arr = np.array(y_true_list)
    logit_pred_arr = np.array(p_pred_list)
    
    # --- 수정된 부분: Logit 역변환 (Sigmoid) ---
    # 1. Sigmoid 함수 적용하여 0~1 사이 비율(ratio)로 변환
    pred_ratio = 1 / (1 + np.exp(-logit_pred_arr))
    # 2. 비율을 Inhibition(%)으로 변환
    y_pred_arr = pred_ratio * 100
    # ---
    
    y_pred_arr = np.clip(y_pred_arr, 0, 100)
    score = competition_score(y_true_arr, y_pred_arr)
    return score, y_true_arr, y_pred_arr

@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    preds_list, ids_list = [], []
    for batch in loader:
        batch = batch.to(device)
        predictions = model(batch, cfg.EPOCHS) 
        
        preds_list.extend(predictions.cpu().numpy())
        
        if hasattr(batch, 'id'):
            ids_list.extend(batch.id)
        else:
            num_graphs_in_batch = batch.num_graphs
            ids_list.extend([f"unknown_{i}" for i in range(num_graphs_in_batch)])
    logit_preds = np.array(preds_list)
    pred_ratio = 1 / (1 + np.exp(-logit_preds))
    inhib_preds = pred_ratio * 100
    
    submission_df = pd.DataFrame({'ID': ids_list, 'Inhibition': inhib_preds})
    original_ids = loader.dataset.df['ID'].tolist()
    submission_df = submission_df.set_index('ID').reindex(original_ids).reset_index()
    
    return submission_df

In [ ]:
print("--- 2. Loading data and starting preprocessing ---")
train_df = pd.read_csv('total.csv')
test_df = pd.read_csv('test.csv')

# --- 2-1. LDS 효과 시각화 (Logit 변환 기반으로 수정) ---
print("\n--- 2-1. Visualizing the effect of Label Distribution Smoothing (LDS) ---")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 한글 폰트 설정
try:
    font_name = 'Malgun Gothic'; plt.rc('font', family=font_name)
except:
    font_name = 'NanumGothic'; plt.rc('font', family=font_name)
plt.rcParams['axes.unicode_minus'] = False
# 1. 원본 Inhibition 값을 Logit 값으로 변환
temp_inhibition = torch.tensor(train_df['Inhibition'].values, dtype=torch.float32)
temp_ratio = torch.clamp(temp_inhibition / 100.0, cfg.EPSILON, 1 - cfg.EPSILON)
logit_values = torch.log(temp_ratio / (1 - temp_ratio))

# 2. 히스토그램을 위한 파라미터 계산
min_v, max_v = int(logit_values.min().floor()), int(logit_values.max().ceil())
if min_v == max_v: max_v += 1
num_bins = max_v - min_v + 1

# 3. 원본 라벨 분포 (히스토그램) 계산 (np.histogram 사용)
# np.histogram의 bins 인자는 경계값 배열이므로, num_bins+1 개의 경계 필요
original_dist, bin_edges = np.histogram(logit_values.numpy(), bins=num_bins, range=(min_v, max_v))
# 시각화를 위한 x축 값 (각 막대의 중심)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# 4. LDS 커널을 적용한 유효 라벨 분포 계산 (torch.histc 사용)
label_dist_tensor = torch.histc(logit_values, bins=num_bins, min=min_v, max=max_v)
lds_kernel = get_lds_kernel_window(cfg.LDS_KERNEL, cfg.LDS_KS, cfg.LDS_SIGMA)
eff_label_dist = F.conv1d(label_dist_tensor.unsqueeze(0).unsqueeze(0), lds_kernel.unsqueeze(0).unsqueeze(0), padding=(cfg.LDS_KS-1)//2).squeeze().numpy()

# 5. 시각화
plt.figure(figsize=(14, 7))
# x축 데이터로 bin_centers를 사용하여 길이를 맞춤
plt.bar(bin_centers, original_dist, width=0.8, alpha=0.7, label='Original Label Distribution', color='skyblue')
plt.plot(bin_centers, eff_label_dist, 'r-o', label='Effective Label Distribution (after LDS)', linewidth=2, markersize=8)
plt.xlabel('Logit Value (Transformed Inhibition)')
plt.ylabel('Number of Samples')
plt.title('Effect of Label Distribution Smoothing (LDS)')
plt.legend()
plt.grid(axis='y', linestyle='--')
plt.tight_layout()
plt.savefig('lds_effect_visualization.png')
plt.show()
print("LDS effect visualization saved as 'lds_effect_visualization.png'\n")

# --- 2-2. 기술자(Descriptors) 계산 및 전처리 ---
print("--- 2-2. Calculating and preprocessing descriptors ---")
train_descriptors = calculate_descriptors(train_df['Canonical_Smiles'])
test_descriptors = calculate_descriptors(test_df['Canonical_Smiles'])

# 결측값 처리 (평균값으로 대체)
imputer = SimpleImputer(strategy='mean')
train_descriptors = imputer.fit_transform(train_descriptors)
test_descriptors = imputer.transform(test_descriptors)

# 스케일링 (중요!)
scaler = StandardScaler()
train_descriptors = scaler.fit_transform(train_descriptors)
test_descriptors = scaler.transform(test_descriptors)
print("Descriptors calculated and scaled.\n")


# --- 2-3. 데이터셋 및 데이터로더 생성 ---
print("--- 2-3. Creating Datasets and DataLoaders ---")
# MolDataset 생성 시, 계산된 기술자 배열을 전달
full_dataset = MolDataset(train_df, train_descriptors, cfg, is_test=False)
train_size = int(cfg.TRAIN_RATIO * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
test_dataset = MolDataset(test_df, test_descriptors, cfg, is_test=True)

# DataLoader 생성 (변경 없음)
train_loader = PyGDataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader = PyGDataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)
test_loader = PyGDataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)
print(f"Train: {len(train_dataset)}, Validation: {len(val_dataset)}, Test: {len(test_dataset)}\n")

In [ ]:
print(f"--- 3. Initializing model and optimizer on device: {cfg.DEVICE} ---")
model = Pignet(cfg).to(cfg.DEVICE)

# --- 차등 학습률(Differential Learning Rates) 설정 ---
# 1. 사전 학습된 인코더 파라미터 그룹
encoder_params = model.gnn_encoder.parameters()
# 2. 새로 추가된 예측기(FDS, MLP) 파라미터 그룹
predictor_params = list(model.fds.parameters()) + list(model.predictor_head.parameters())

# 옵티마이저에 각 그룹별로 다른 학습률을 지정
optimizer = torch.optim.AdamW([
    {'params': encoder_params, 'lr': 1e-5}, # 인코더는 매우 낮은 LR로 미세조정
    {'params': predictor_params, 'lr': 1e-4} # 예측기는 상대적으로 높은 LR로 빠르게 학습
], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max',      # Val Score는 높을수록 좋으므로 'max'
    factor=0.5,      # 성능 개선이 없으면 LR을 50%로 줄임
    patience=7       # 7 에포크 동안 Val Score가 최고치를 경신 못하면 LR 감소
)


In [ ]:
# 기존 학습 루프 코드를 아래 코드로 교체해주세요.

# --- 학습 루프 ---
print("\n--- 4. Starting training loop with LDS & FDS ---")
best_score = 0
epochs_no_improve = 0
for epoch in range(cfg.EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, epoch, cfg.DEVICE)
    
    # ==================== 수정된 핵심 부분 (순서 변경) ====================
    # 1. 먼저 FDS 효과를 시각화하고,
    # 2. 그 다음에 통계를 업데이트하고 리셋합니다.
    # ====================================================================

    # FDS가 시작되는 첫 에포크에 효과를 시각화
    if epoch == cfg.FDS_START_UPDATE_EPOCH:
        print("\n--- Visualizing the effect of Feature Distribution Smoothing (FDS) ---")
        
        # FDS 모듈에서 '현재' 통계 정보 가져오기 (last_epoch이 아님)
        running_mean = model.fds.running_mean.cpu().numpy()
        # 스무딩 함수를 직접 호출하여 스무딩된 결과 확인
        smoothed_mean = model.fds._smooth(model.fds.running_mean, model.fds.bucket_counts).cpu().numpy()
        
        # 시각화를 위해 하나의 특성 차원 선택 (예: 0번째 차원)
        # feature_dim_to_plot = 0
        
        # 데이터가 있는 버킷만 시각화
        # valid_buckets = np.where(model.fds.bucket_counts.cpu().numpy() > 0)[0]
        
        # if len(valid_buckets) > 0: # 데이터가 있는 버킷이 있을 경우에만 시각화
        #     plt.figure(figsize=(14, 7))
        #     plt.plot(valid_buckets, running_mean[valid_buckets, feature_dim_to_plot], 'o--', alpha=0.7, label=f'Original Mean (Feature Dim {feature_dim_to_plot})', color='skyblue')
        #     plt.plot(valid_buckets, smoothed_mean[valid_buckets, feature_dim_to_plot], 'r-', label=f'Smoothed Mean (after FDS)', linewidth=2)
            
        #     plt.xlabel(f'p-value Bucket (Bucket Start: {cfg.FDS_BUCKET_START})')
        #     plt.ylabel(f'Mean value of Feature Dimension {feature_dim_to_plot}')
        #     plt.title(f'Effect of Feature Distribution Smoothing (FDS) at Epoch {epoch}')
        #     plt.legend()
        #     plt.grid(axis='y', linestyle='--')
        #     plt.tight_layout()
        #     plt.savefig('fds_effect_visualization.png')
        #     plt.show()
            
        #     print("FDS effect visualization saved as 'fds_effect_visualization.png'\n")
        # else:
        #     print("No data in FDS buckets to visualize.")

    # 통계 업데이트 (시각화 이후에 호출)
    model.fds.update_last_epoch_stats(epoch)

    # 검증 및 스케줄러 업데이트
    val_score, _, _ = eval_epoch(model, val_loader, epoch, cfg.DEVICE)
    scheduler.step(val_score) 
    
    print(f"Epoch {epoch+1:02d}/{cfg.EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val Score: {val_score:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    if val_score > best_score:
        best_score = val_score
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"   -> New best model saved with score: {best_score:.4f}")
        epochs_no_improve = 0 # 개선되었으므로 카운터 리셋
    else:
        epochs_no_improve += 1 # 개선되지 않았으므로 카운터 증가

    # 조기 종료 조건 확인
    if epochs_no_improve == cfg.EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping triggered after {cfg.EARLY_STOPPING_PATIENCE} epochs without improvement.")
        break # 학습 루프 탈출

In [ ]:
# --- 5. 모델 성능 시각화 ---
print("\n--- 5. Visualizing model performance on validation set ---")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정
try:
    font_name = 'Malgun Gothic'
    plt.rc('font', family=font_name)
except:
    font_name = 'NanumGothic'
    plt.rc('font', family=font_name)
plt.rcParams['axes.unicode_minus'] = False

# 최고의 모델 가중치 로드
model.load_state_dict(torch.load('best_model.pth'))

final_score, y_val_true, y_val_pred = eval_epoch(model, val_loader, cfg.EPOCHS, cfg.DEVICE)
# ========================================================

# 실제값 vs 예측값 산점도
plt.figure(figsize=(10, 6))
plt.scatter(y_val_true, y_val_pred, alpha=0.5, label='예측값')
plt.plot([0, 100], [0, 100], 'r--', label='y=x (이상적인 예측)') # 0~100% 범위로 수정
plt.xlabel('실제 Inhibition (%)')
plt.ylabel('예측 Inhibition (%)')
plt.title(f'PIGNET 모델 검증 성능 (Final Competition Score: {final_score:.4f})')
plt.legend()
plt.grid(True)
plt.xlim(0, 105) # X축 범위 설정
plt.ylim(0, 105) # Y축 범위 설정
plt.savefig('model_performance.png')
print("모델 성능 시각화 저장: model_performance.png")


# 특성 중요도 시각화 (설명 포함)
# ... (이 부분은 이전과 동일하게 유지) ...
if hasattr(model, 'feature_importances_'):
    print("GNN 모델은 feature_importances_ 속성을 직접 지원하지 않습니다.")
else:
    print("\nGNN 모델은 .feature_importances_를 지원하지 않습니다.")
    print("특성 중요도 분석을 위해서는 GNNExplainer와 같은 해석 가능성(XAI) 기법을 사용해야 합니다.")

In [ ]:
print("\n--- 6. Predicting on test set and creating submission file ---")
# 최고의 모델 가중치는 이미 로드됨
submission_df = predict(model, test_loader, cfg.DEVICE)

# 제출 파일 저장
submission_df.to_csv('submission.csv', index=False)
print("submission.csv file has been created successfully.")
print(submission_df.head())